# Asynchronous Programming

## 📖 1. The `async` and `await` keywords

In Python 3.5+, you can define a function as asynchronous using the `async def` keyword.
Inside an `async` function, you can use the `await` keyword to "pause" the execution of that function until the awaited task is complete, allowing the program (the Event Loop) to do other work in the meantime.

In [1]:
import asyncio

async def fetch_data():
    print("Start fetching...")
    await asyncio.sleep(2)  # Simulates waiting for a database or network
    print("Done fetching!")
    return {"data": 42}

# To run an async function outside of FastAPI, you use an Event Loop:
result = asyncio.run(fetch_data())

RuntimeError: asyncio.run() cannot be called from a running event loop

## 📖 2. Async Endpoints in FastAPI

FastAPI fully supports `async def`. If you are calling an asynchronous library (like `httpx` for requests, or an async database driver), you should use `async def`.

In [2]:
# @app.get("/fast-async-route")
# async def get_data():
#     # We can await async functions here!
#     await asyncio.sleep(1)
#     return {"message": "This didn't block the server!"}

## 📖 3. When NOT to use `async def`

If you are using a **synchronous** library (like the standard `time.sleep()`, or synchronous SQLAlchemy, or synchronous `requests`), and you put it inside an `async def` function, **you will block the entire server!**

FastAPI is smart: if you define a route with normal `def` (without `async`), it will run that function in an external threadpool so it doesn't block the main event loop.

In [ ]:
import time

# GOOD: FastAPI puts this in a threadpool.
@app.get("/good-sync")
def good_sync():
    time.sleep(2) 
    return {"message": "I am slow, but I didn't crash the server"}

# BAD! Do not do this. You are lying to FastAPI!
@app.get("/bad-async")
async def bad_async():
    time.sleep(2)  # This BLOCKS the event loop for EVERYONE.
    return {"message": "I just ruined the performance of my API"}

## ⚠️ Common Mistakes
- Using `time.sleep()` inside an `async def` route.
- Forgetting the `await` keyword when calling an async function.
- Trying to use `await` inside a normal `def` function.

## 🏋️ Exercises

Open the `async_api/` folder and inspect `app.py`. Run it and see the console output to understand blocking vs non-blocking code.
Then, complete the exercises in `exercises/03_exercises.py`.